# Phase 1 / Stage 2 - Plant disease detection (ResNet-18)

Capstone Project 7, Group 4. This notebook trains, evaluates and publishes the disease classifier
and hands it over to Phase 5.

**Pipeline (this notebook)**

```
PlantVillage zip on SSD
  -> 3b EDA (class-count chart, augmentation preview, per-class progress)
  -> 5  Stage 1 train: PlantVillage only, 38 classes, no PlantDoc  -> tag v1.0
  -> 7b Stage 2: Cropped-PlantDoc prep
       -> mixed epoch (50% PV uniform + 50% PlantDoc 1/sqrt(freq))
       -> calibrate on PlantDoc val -> one-shot final test -> tag v1.1
```

Training, sampling, and Hub upload live in `model/*.py`. This notebook is the control panel:
progress logs, charts, and the knobs cell after Hugging Face login. Default knobs **skip**
retraining so Run-all does not retag the live model.

**What each stage does**

1. Loads PlantVillage (color) from a Drive zip copied onto the Colab SSD (falls back to `kagglehub`).
2. Fine-tunes `microsoft/resnet-18` (stage 1) and tags `v1.0` (section 5 + 8).
3. Reports PlantVillage accuracy, per-class F1, confusion matrix, top-3 and a threshold sweep.
4. **Legacy** section 7: the 50-image real-world set through the live-app chain (used to choose v1.0's threshold).
5. Pushes to **https://huggingface.co/neghani/plantvillage-resnet18** (shell equivalent: [`model/train_resnet18.sh`](../model/train_resnet18.sh)).
   Three Hub steps: **weights**, **reports/card**, **release tag**. `shared/release.json` is **not**
   edited here; the live API pin stays until that file is changed on purpose.
6. **Stage 2 (section 7b):** continues from `v1.0` on a 50/50 mix of PlantVillage and Cropped-PlantDoc
   (Singh et al. 2020, CC BY 4.0; 27 of 38 classes). PlantDoc classes are sampled with weight
   `1/sqrt(freq)`. Diagnostics and calibration use **validation only**.
   The final test runs **once** per revision (guard on Hub/Drive). Publish `v1.1` only if PlantDoc val
   macro-F1 beats v1.0 and PlantVillage accuracy drops by at most 2 points.

**How to test (local vs Colab)**

- **Local (CPU):** `python -m model.train --smoke` (synthetic data, ~2 min, writes only `model/outputs/smoke/`, never the final-test guard). Then `git push`.
- **Colab (GPU):** do **not** `files.upload()` code. Re-run the sync cell. It checks out a **pinned commit**, prints `CODE_COMMIT`, and leaves `model/outputs/` and images alone.
- **Real-world images:** git-ignored. Keep a zip on Drive (`--real-world-set-from`).

**Before you start (Colab)**

- Runtime -> Change runtime type -> **T4 GPU** (or better).
- Colab secrets: `HF_TOKEN` (write token). Optional: `KAGGLE_USERNAME` and `KAGGLE_KEY`.
- Optional: put `MyDrive/plant-disease/plantvillage.zip` on Drive so section 3 copies it to the local SSD instead of downloading from Kaggle.
- After you push stage-2 code, set `REPO_COMMIT` in the sync cell to that SHA so Colab cannot silently drift.

In [ ]:
!nvidia-smi -L || echo "No GPU - training will be very slow; switch the runtime to a GPU."

## 1. Sync the repository (pinned commit)

Fetches `notebooks/colab_sync.py` from **main**, then `git fetch` + `git reset --hard`
to `origin/main` (or a SHA if you set `ALLOW_BRANCH_OVERRIDE=1`). That replaces a bare `git pull`,
which fails when the Colab clone and GitHub have diverged. A leftover `REPO_BRANCH` in the runtime
is ignored unless you opt in. Prints `CODE_COMMIT` for `run_info.json`.

After you push a local fix, **re-run this cell**. Do not upload `.py` files. Do not `git pull` in
another cell.

In [ ]:
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

# Default branch is main. A leftover Colab REPO_BRANCH (e.g. Phase4) must not win.
# Set ALLOW_BRANCH_OVERRIDE=1 only when you intentionally want another ref.
_OVERRIDE = os.environ.get("ALLOW_BRANCH_OVERRIDE", "").strip().lower() in ("1", "true", "yes")
if _OVERRIDE:
    REPO_BRANCH = os.environ.get("REPO_BRANCH", "main")
    REPO_COMMIT = os.environ.get("REPO_COMMIT", "")
else:
    os.environ.pop("REPO_BRANCH", None)
    os.environ.pop("REPO_COMMIT", None)
    REPO_BRANCH = "main"
    REPO_COMMIT = ""
os.environ["REPO_BRANCH"] = REPO_BRANCH
print("REPO_BRANCH", REPO_BRANCH, "REPO_COMMIT", REPO_COMMIT or "(branch tip)",
      "ALLOW_BRANCH_OVERRIDE", _OVERRIDE)

REPO = "neghani/iith-group4-capstone-plant-leaf-disease-detection_v1"
ref = REPO_COMMIT or REPO_BRANCH
RAW = f"https://raw.githubusercontent.com/{REPO}/{ref}/notebooks/colab_sync.py"
helper = Path("/tmp/colab_sync.py")
try:
    urllib.request.urlretrieve(RAW, helper)
    print("Fetched", RAW)
except Exception as exc:
    local = Path("notebooks/colab_sync.py")
    if not local.exists():
        raise RuntimeError("Could not fetch colab_sync.py from GitHub; clone the repo first.") from exc
    helper = local
    print("Using local", helper, "because raw fetch failed:", exc)

spec = importlib.util.spec_from_file_location("colab_sync", helper)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
argv = ["--branch", REPO_BRANCH]
if REPO_COMMIT:
    argv += ["--commit", REPO_COMMIT]
mod.main(argv)

root = Path(os.environ.get("REPO_ROOT") or "/content/iith-group4-capstone-plant-leaf-disease-detection_v1")
os.chdir(root)
os.environ["REPO_ROOT"] = str(root)
os.environ["PYTHONPATH"] = str(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
os.environ["CODE_COMMIT"] = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True).strip()
origin_ref = subprocess.check_output(
    ["git", "rev-parse", f"origin/{REPO_BRANCH}"], cwd=root, text=True
).strip()
print("cwd", os.getcwd())
print("CODE_COMMIT", os.environ["CODE_COMMIT"])
if not _OVERRIDE and os.environ["CODE_COMMIT"] != origin_ref:
    raise SystemExit(
        f"section 1 is at {os.environ['CODE_COMMIT'][:12]}, not origin/{REPO_BRANCH} "
        f"({origin_ref[:12]}). Re-run this cell. Do not git pull in another cell."
    )
import importlib
cs = importlib.import_module("notebooks.colab_sync")
importlib.reload(cs)
if getattr(cs, "DEFAULT_BRANCH", None) != "main":
    raise SystemExit(
        f"colab_sync.DEFAULT_BRANCH is {getattr(cs, 'DEFAULT_BRANCH', None)!r}, expected 'main'. "
        "Push notebooks/colab_sync.py and re-run this cell."
    )
if not hasattr(cs, "DEFAULT_DRIVE_PV_ZIP"):
    raise SystemExit(
        "section 1 reset to a colab_sync.py without DEFAULT_DRIVE_PV_ZIP. "
        "Push notebooks/colab_sync.py and re-run this cell. Do not git pull in another cell."
    )
print("PlantVillage staging helper OK:", cs.DEFAULT_DRIVE_PV_ZIP)

Optional: replace **one** file after a local push (still from GitHub, always overwrite). This does
**not** update `colab_sync.py` and is not a substitute for section 1. After a helper or notebook
change, re-run section 1. Do not `files.upload()`.

In [ ]:
# Do not use this instead of section 1. --file does not update notebooks/colab_sync.py.
# After a helper change, re-run section 1 (fetch + reset --hard).
from notebooks.run import run

# run("notebooks/colab_sync.py", "--skip-install", "--branch", "main", "--file", "model/train.py")
print("Skipped --file replace. Re-run section 1 to update the clone from GitHub.")


## 2. Log in to Hugging Face (and Kaggle if needed)

The push target is fixed to `neghani/plantvillage-resnet18`. If you run this with a different account, set `HF_MODEL_ID` to your own namespace below. Tokens come from Colab secrets.

In [ ]:
import os
from huggingface_hub import HfApi, login

try:
    from google.colab import userdata  # type: ignore
    token = userdata.get("HF_TOKEN")
    for key in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        try:
            val = userdata.get(key)
        except Exception:
            val = None
        if val:
            os.environ[key] = val
except Exception:
    token = os.environ.get("HF_TOKEN")
assert token, "Add HF_TOKEN to Colab secrets (enable 'Notebook access') or export HF_TOKEN."
login(token=token, add_to_git_credential=False)
os.environ["HF_TOKEN"] = token

HF_MODEL_ID = os.environ.get("HF_MODEL_ID", "neghani/plantvillage-resnet18")
me = HfApi().whoami()["name"]
expected_owner = HF_MODEL_ID.split("/")[0]
assert me == expected_owner or expected_owner in [o["name"] for o in HfApi().whoami().get("orgs", [])], (
    f"Logged in as '{me}' but HF_MODEL_ID is '{HF_MODEL_ID}'. "
    f"Either log in as '{expected_owner}' or set os.environ['HF_MODEL_ID'] = '{me}/plantvillage-resnet18' before this cell."
)
os.environ["HF_MODEL_ID"] = HF_MODEL_ID
print("Logged in as", me, "-> will push to", HF_MODEL_ID)
print("Kaggle secrets:", "yes" if os.environ.get("KAGGLE_USERNAME") else "not set (public dataset usually works anyway)")

## 2b. Session knobs (the only place to turn a stage on)

`RUN_STAGE1` and `RUN_STAGE2` stay **False** so Run-all does not retrain or retag the live Hub model.
Set a flag True only when you intend to write `v1.0` or `v1.1`. Later cells read these names; do not
redefine the flags further down.

In [ ]:
from notebooks.run import apply_knobs, print_session_plan

# Flip a flag only for this training session. Defaults stay False so Run-all does not retag the Hub.
RUN_STAGE1 = False  # PlantVillage-only train, evaluate --push, publish.py --tag v1.0
RUN_STAGE2 = False  # mixed PlantVillage+PlantDoc train, final_test, publish.py --tag v1.1

STAGE1_EPOCHS = 5
STAGE1_LR = 5e-4
STAGE1_BATCH = 64

PD_FRACTION = 0.5          # PlantDoc share of each mixed epoch; rest is PlantVillage
EPOCH_SAMPLES = 4000       # draws per mixed epoch
STAGE2_EPOCHS = 8
STAGE2_LRS = ["1e-4", "5e-5"]  # Drive folders lr1e-4 / lr5e-5; keep these names to resume
STAGE2_BATCH = 64

apply_knobs(
    RUN_STAGE1=RUN_STAGE1,
    RUN_STAGE2=RUN_STAGE2,
    STAGE1_EPOCHS=STAGE1_EPOCHS,
    STAGE1_LR=STAGE1_LR,
    STAGE1_BATCH=STAGE1_BATCH,
    PD_FRACTION=PD_FRACTION,
    EPOCH_SAMPLES=EPOCH_SAMPLES,
    STAGE2_EPOCHS=STAGE2_EPOCHS,
    STAGE2_LRS=STAGE2_LRS,
    STAGE2_BATCH=STAGE2_BATCH,
)
print_session_plan()

## 3. Load PlantVillage (color) onto local SSD

Prefer a single zip on Drive (`MyDrive/plant-disease/plantvillage.zip`). This cell copies it to
`/content/plantvillage.zip` and unzips onto the Colab SSD so training does not read through the
Drive mount. If the zip is missing, it falls back to `kagglehub` (also on the VM disk). The zip
may be the Kaggle root or the `color/` imagefolder.

In [ ]:
import os
from pathlib import Path
from notebooks.run import ensure_repo

ROOT = ensure_repo()
import notebooks.colab_sync as cs
if not hasattr(cs, "DEFAULT_DRIVE_PV_ZIP"):
    raise SystemExit(
        "colab_sync.py on disk is old. Re-run section 1 only (git fetch + reset --hard). "
        "Do not git pull in another cell. Do not paste files."
    )
cs.assert_pv_staging_helper(cs)
from notebooks.colab_sync import (
    DEFAULT_DRIVE_PV_ZIP,
    DEFAULT_LOCAL_PV_DIR,
    DEFAULT_LOCAL_PV_ZIP,
    maybe_mount_drive,
    stage_zip_locally,
)
from model.common import resolve_data_dir

maybe_mount_drive()
drive_zip = Path(DEFAULT_DRIVE_PV_ZIP)
if drive_zip.exists():
    staged = stage_zip_locally(drive_zip, Path(DEFAULT_LOCAL_PV_DIR), Path(DEFAULT_LOCAL_PV_ZIP))
    DATA_DIR = str(resolve_data_dir(staged))
    print("Staged PlantVillage from Drive zip ->", DATA_DIR)
else:
    import kagglehub
    root = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
    print("No Drive zip at", drive_zip, "; downloaded to", root)
    DATA_DIR = str(resolve_data_dir(root))

os.environ["DATA_DIR"] = DATA_DIR
classes = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
print(len(classes), "classes, e.g.", classes[:5])
print("images:", sum(len(os.listdir(os.path.join(DATA_DIR, c))) for c in classes))
print("DATA_DIR", DATA_DIR)

## 3b. Data sanity, EDA, and augmentation preview

Opens every PlantVillage image the trainer will use. Prints `class k/38 ..., N images opened` as it
goes so the wait is not a blank cell. Stops on a hard failure (wrong class set, empty class,
unreadable file) so section 5 does not die mid-epoch. Warnings (imbalance, small images,
near-duplicates in a sample) do not block training and do not rewrite splits.

Writes `model/outputs/plantvillage_eda.json`, `plantvillage_eda.md`, `plantvillage_class_counts.png`,
and an augmentation contact sheet. The next cell shows those charts. This cell does not start a new run.

In [ ]:
import os
from notebooks.run import ensure_repo, require_env, run

ensure_repo()
require_env("DATA_DIR")
run("model/eda.py", "--data_dir", os.environ["DATA_DIR"])


In [ ]:
import json
from pathlib import Path
from IPython.display import Image, Markdown, display

eda = Path("model/outputs")
payload_path = eda / "plantvillage_eda.json"
payload = json.loads(payload_path.read_text()) if payload_path.exists() else {}
md = eda / "plantvillage_eda.md"
if md.exists():
    display(Markdown(md.read_text()))
rows = payload.get("per_class") or []
if rows:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, max(4.0, 0.28 * len(rows))))
    ax.barh([r["name"] for r in rows], [r["n"] for r in rows], color="#3d7a46")
    ax.set_xlabel("Images")
    ax.set_title("PlantVillage class counts")
    ax.invert_yaxis()
    fig.tight_layout()
    display(fig)
    plt.close(fig)
elif (eda / "plantvillage_class_counts.png").exists():
    display(Image(filename=str(eda / "plantvillage_class_counts.png"), width=900))
preview = eda / "plantvillage_aug_preview.png"
if preview.exists():
    print("Augmentation preview (left: original, right: current train transforms)")
    display(Image(filename=str(preview), width=800))
for w in payload.get("warnings") or []:
    print("WARN:", w["message"])
if payload and not payload.get("ok", True):
    print("EDA reported errors; do not start section 5.")


## 4. Quick smoke test (optional)

**Local (no downloads):** `python -m model.train --smoke` — synthetic images, 1–2 CPU steps, writes only
`model/outputs/smoke/`, never the final-test guard.

**Colab (optional, ~2 min):** tomato classes only, 60 images per class, 1 epoch on real PlantVillage.
Writes `model/outputs/tomato_smoke/` so it does not collide with the synthetic smoke dir. Nothing is pushed.

In [ ]:
import os
from notebooks.run import ensure_repo, require_env, run
from model.common import gpu_train_flags

ensure_repo()
require_env("DATA_DIR")
run("model/train.py",
    "--data_dir", os.environ["DATA_DIR"], "--crop_filter", "Tomato", "--max_samples_per_class", "60",
    "--output_dir", "model/outputs/tomato_smoke",
    "--do_train", "--do_eval", "--num_train_epochs", "1",
    "--per_device_train_batch_size", "32", "--per_device_eval_batch_size", "64",
    "--learning_rate", "5e-4", *gpu_train_flags(), "--report_to", "none", "--save_strategy", "no", "--logging_steps", "5")
run("model/evaluate.py", "--model", "model/outputs/tomato_smoke")


## 5. Full training run

PlantVillage only: all 38 classes (~54k images), ImageNet ResNet-18, **no PlantDoc**. The cell
below prints the recipe (epochs, learning rate, output folder) before `train.py` starts.

**Skipped unless** `RUN_STAGE1 = True` in the **section 2b knobs cell**. Set it True to train ResNet stage 1 and tag `v1.0`.
Checkpoints go to **Drive** (`MyDrive/plant-disease/stage1/...`) so a disconnect does not wipe the run.
Re-running with the flag **resumes**, or **skips** if the run already finished. Pass
`--overwrite_output_dir` only to throw it away and start over.


In [ ]:
import os
from pathlib import Path
from notebooks.run import ensure_repo, flag_on, knob, print_publish_checklist, require_env, run
from model.common import gpu_train_flags

ensure_repo()
require_env("DATA_DIR", "HF_MODEL_ID")
drive = Path("/content/drive/MyDrive/plant-disease")
OUTPUT_DIR = str(drive / "stage1/plantvillage-resnet18") if drive.exists() else "model/outputs/plantvillage-resnet18"
os.environ["OUTPUT_DIR"] = OUTPUT_DIR
print("=== Stage 1: PlantVillage only ===")
print("classes: 38  (PlantDoc is not in this run)")
print("base model: microsoft/resnet-18")
print("epochs:", knob("STAGE1_EPOCHS"), "  learning_rate:", knob("STAGE1_LR"),
      "  batch:", knob("STAGE1_BATCH"))
print("output:", OUTPUT_DIR)
print("RUN_STAGE1", flag_on("RUN_STAGE1"), "(set in section 2b knobs)")
print_publish_checklist()
if not flag_on("RUN_STAGE1"):
    print("Skipping section 5 train. Set RUN_STAGE1 = True in the knobs cell to train ResNet stage 1 (5 epochs) and tag v1.0. Do not evaluate an empty Drive folder.")
else:
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    run("model/train.py",
        "--model_name_or_path", "microsoft/resnet-18",
        "--data_dir", os.environ["DATA_DIR"],
        "--output_dir", OUTPUT_DIR,
        "--resume_from_checkpoint", "auto",
        "--do_train", "--do_eval",
        "--num_train_epochs", knob("STAGE1_EPOCHS"),
        "--per_device_train_batch_size", knob("STAGE1_BATCH"), "--per_device_eval_batch_size", "128",
        "--learning_rate", knob("STAGE1_LR"), "--weight_decay", "0.01", "--warmup_steps", "500",
        "--lr_scheduler_type", "cosine",
        *gpu_train_flags(),
        "--eval_strategy", "epoch", "--save_strategy", "epoch", "--save_total_limit", "2",
        "--load_best_model_at_end", "--metric_for_best_model", "macro_f1", "--greater_is_better", "True",
        "--logging_steps", "50", "--report_to", "none", "--seed", "42",
        "--push_to_hub", "--hub_model_id", os.environ["HF_MODEL_ID"], "--hub_strategy", "end")


## 6. Evaluate on the held-out test split and build the model card

Produces `reports/` (metrics.json, classification report, confusion matrix, threshold sweep) and fills `README.md`; `--push` uploads both to the Hub.

**Skipped unless** `RUN_STAGE1 = True` in the section 2b knobs cell. This cell evaluates the local Drive folder from section 5. Train first, or skip to section 7b.

In [ ]:
import os
from notebooks.run import ensure_repo, flag_on, require_env, run

ensure_repo()
if not flag_on("RUN_STAGE1"):
    print("Skipping section 6 evaluate/push. Set RUN_STAGE1 = True in the knobs cell and train before evaluating the local Drive folder.")
else:
    require_env("OUTPUT_DIR", "HF_MODEL_ID")
    run("model/evaluate.py", "--model", os.environ["OUTPUT_DIR"], "--hub_model_id", os.environ["HF_MODEL_ID"], "--push")


In [ ]:
from notebooks.run import flag_on

if not flag_on("RUN_STAGE1"):
    print("Skipping section 6 reports: no local evaluate ran.")
else:
    from IPython.display import Image, display
    import json, os
    display(Image(filename=os.path.join(os.environ["OUTPUT_DIR"], "reports", "confusion_matrix.png"), width=900))
    m = json.load(open(os.path.join(os.environ["OUTPUT_DIR"], "reports", "metrics.json")))
    print({k: m[k] for k in ("accuracy", "macro_f1", "top3_accuracy", "recommended_confidence_threshold")})

## 7. Real-world test set (legacy v1.0 path)

Used to choose crop strategy B and threshold 0.85 for `v1.0`. Stage 2 (section 7b) does **not**
retune on this set: the 50 images are in the final test only, excluded from training, and the
threshold is chosen on PlantDoc **validation**. Keep this section for reproducing the v1.0 card.

The labelled set lives in `model/data/real_world_set/` (`labels.csv` with a `tune` / `final` split,
`sources.csv`, optional `card_notes.md`, and `images/`). The GitHub repo is public, so the images are
git-ignored and come from Drive: `MyDrive/plant-disease/real_world_set.zip` (zip the whole folder).

**New or changed set:** replace that zip on Drive and re-run this section. The next cell wipes the
local set and copies the zip in. The eval cell then runs the live-app chain (API quality gate + leaf
crop -> model) on `tune` and `final`, and compares a fingerprint of `labels.csv` + every image, the
model weights and the threshold with `reports/real_world_set/<split>/run.json` on the Hub:

- something changed (or the card was overwritten) -> results are uploaded and the **Hub README's
  real-world section and `model-index` are regenerated** from the `final` split;
- nothing changed -> it prints `unchanged` and pushes nothing.

Set `FORCE_PUBLISH = True` to republish anyway.

In [ ]:
# images/ is git-ignored. Prefer the Drive zip; otherwise rebuild from Cropped-PlantDoc (seed 42).
import os, subprocess, sys
from pathlib import Path

root = Path(os.environ.get("REPO_ROOT") or "/content/iith-group4-capstone-plant-leaf-disease-detection_v1")
os.chdir(root)
REAL_WORLD_SET_SRC = Path("/content/drive/MyDrive/plant-disease/real_world_set.zip")
dest = root / "model" / "data" / "real_world_set"
plantdoc = root / "model" / "data" / "plantdoc_src" / "train"

if REAL_WORLD_SET_SRC.exists():
    subprocess.check_call([sys.executable, "notebooks/colab_sync.py", "--skip-install",
                           "--no-reset", "--branch", "main",
                           "--real-world-set-from", str(REAL_WORLD_SET_SRC)], cwd=root)
elif plantdoc.is_dir():
    print("No Drive zip; rebuilding the 50-image set from", plantdoc.parent)
    subprocess.check_call([sys.executable, "model/build_real_world_set.py", str(plantdoc.parent), str(dest)], cwd=root)
else:
    raise SystemExit(
        f"No images at {dest}/images and no zip at {REAL_WORLD_SET_SRC}.\n"
        "Put real_world_set.zip on Drive, or run section 7b prep first (clones PlantDoc)."
    )
n_img = sum(1 for p in dest.joinpath("images").rglob("*") if p.is_file())
print(f"real_world_set images: {n_img}")
assert n_img >= 50, f"expected 50 images under {dest}/images, got {n_img}"

In [ ]:
import os
from pathlib import Path
from notebooks.run import ensure_repo, require_env, run

ensure_repo()

FORCE_PUBLISH = False

# Model under test: this session's run if it exists, else the published release.
out_dir = Path(os.environ.get("OUTPUT_DIR", "model/outputs/plantvillage-resnet18"))
if (out_dir / "model.safetensors").exists():
    EVAL_MODEL, EVAL_REV = str(out_dir), ""
else:
    # Published ResNet stage-1 tag. Id and revision are set together here; HF_MODEL_REVISION is not read.
    EVAL_MODEL, EVAL_REV = os.environ["HF_MODEL_ID"], "v1.0"
# Crop strategy + threshold chosen on the tune split by model/eval_strategies.py (= api/app/config.py).
STRATEGY, THRESHOLD = "B", 0.85
print(f"Testing {EVAL_MODEL}{'@' + EVAL_REV if EVAL_REV else ''} with strategy {STRATEGY} at threshold {THRESHOLD}")

rev_flag = f"--revision {EVAL_REV}" if EVAL_REV else ""
change_flag = "" if FORCE_PUBLISH else "--only_if_changed"
# tune first (thresholds are chosen there), final last: only final rewrites the Hub README.
require_env("HF_MODEL_ID")
for split in ("tune", "final"):
    args = [
        "model/eval_real_world_set.py",
        "--set_dir", "model/data/real_world_set", "--split", split,
        "--predictor", "app_predictor:diagnose", "--strategy", STRATEGY, "--threshold", str(THRESHOLD),
        "--model", EVAL_MODEL, "--hub_model_id", os.environ["HF_MODEL_ID"],
        "--out", f"model/outputs/real_world/{split}", "--push",
    ]
    if EVAL_REV:
        args += ["--revision", EVAL_REV]
    if not FORCE_PUBLISH:
        args.append("--only_if_changed")
    run(*args)


In [ ]:
from IPython.display import Markdown, display
report = Path("model/outputs/real_world/final/report.md")
display(Markdown(report.read_text()) if report.exists() else Markdown("_No final report yet._"))
print(f"Hub card: https://huggingface.co/{os.environ['HF_MODEL_ID']}")

### Optional: ad-hoc photo folder (not published to the card)

Loose photos in `model/data/real_world/<ClassName>/` from `MyDrive/plant-disease/real_world.zip`,
scored full-frame by `model/real_world_eval.py`. Kept for quick checks; the card's real-world section
comes from the labelled set above.

In [ ]:
# REAL_WORLD_SRC = "/content/drive/MyDrive/plant-disease/real_world.zip"
# !python notebooks/colab_sync.py --skip-install --real-world-from $REAL_WORLD_SRC
# !python model/real_world_eval.py --model "$OUTPUT_DIR" --hub_model_id "$HF_MODEL_ID"

## 7b. Stage 2: fine-tune on Cropped-PlantDoc, calibrate, test once

`v1.0` was trained on PlantVillage only (one leaf, plain background). This stage continues from those
weights on a **50/50 mix** of PlantVillage and [Cropped-PlantDoc](https://github.com/pratikkayal/PlantDoc-Dataset)
(Singh et al. 2020, CC BY 4.0), pinned at `5467f6012d78d1c446145d5f582da6096f852ae8`. 27 of 38 PlantVillage
classes appear in PlantDoc; the other 11 stay lab-only. Spider-mites is dropped.

**Rules**

- Decisions use **PlantDoc validation only**. Do not look at the final test until `final_test.py`.
- The mixed-epoch recipe cell prints PlantDoc `1/sqrt(freq)` class weights and expected draws before training starts.
- Report sample sizes next to every percentage.
- If the final pass rate is still below 50%, report it honestly. Do not change the threshold or the data after seeing it.
- Checkpoints go to Drive every epoch (`--resume_from_checkpoint auto`). Small artifacts upload to Hub `stage2-runs` and Drive after each step.
- The final-test **guard** is `hub:$HF_MODEL_ID` (JSON named by revision on `stage2-runs`) or `drive:.../final_test_guard`.

**Needs this session:** section 1 (pinned `CODE_COMMIT`), section 2 (HF token), section 3 (`DATA_DIR`),
section 7's real-world set (excluded from train by `original_path`).

Train, compare, calibrate, final-test, and `publish --tag v1.1` are **skipped unless**
`RUN_STAGE2 = True` in the **section 2b knobs cell**. Set that flag True to train ResNet v1.1 (PlantDoc mix). This notebook does not edit shared/release.json.

In [ ]:
import os, subprocess
from pathlib import Path
from notebooks.run import ensure_repo, flag_on, run_mod, stage2_lrs
from notebooks.colab_sync import DEFAULT_BRANCH, in_colab, maybe_mount_drive

ROOT = ensure_repo()
if DEFAULT_BRANCH != "main":
    raise SystemExit(
        f"colab_sync.DEFAULT_BRANCH is {DEFAULT_BRANCH!r}. Re-run section 1 with main."
    )
os.environ["CODE_COMMIT"] = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
print("cwd", os.getcwd())
print("CODE_COMMIT", os.environ["CODE_COMMIT"])
if not os.environ.get("DATA_DIR"):
    print("NOTE: DATA_DIR is unset. Re-run section 3 before the train cells (prep can still run).")

print("RUN_STAGE2", flag_on("RUN_STAGE2"), "(set in section 2b knobs; train/compare/calibrate/publish skip unless True)")

maybe_mount_drive()
drive_root = Path("/content/drive/MyDrive")
if drive_root.is_dir():
    DRIVE = "/content/drive/MyDrive/plant-disease"
    STAGE2 = f"{DRIVE}/stage2"
elif in_colab():
    raise SystemExit(
        "Google Drive is not mounted at /content/drive/MyDrive. "
        "Mount Drive, then re-run this cell. Do not write stage2 onto a fake local path."
    )
else:
    STAGE2 = "model/outputs/stage2"
    print("WARNING: Drive not mounted; writing stage2 to", STAGE2)
RUN = "stage2"
ARTIFACTS = "model/outputs"
STAGE2_LR_DIRS = [f"{STAGE2}/lr{s}" for s in stage2_lrs()]
os.environ["STAGE2"] = STAGE2
os.environ["STAGE2_LR_DIRS"] = ",".join(STAGE2_LR_DIRS)
print("STAGE2 folders", STAGE2_LR_DIRS)
os.makedirs(STAGE2, exist_ok=True)

def upload(step, *paths):
    missing = [p for p in paths if not Path(p).exists()]
    if missing:
        raise SystemExit(f"upload {step}: missing {missing} (the previous step did not finish)")
    run_mod("model.artifacts", "--step", step, "--run", RUN,
            "--hub", os.environ["HF_MODEL_ID"], "--drive", f"{STAGE2}/artifacts", *paths)

# Clones Cropped-PlantDoc if needed, excludes real_world_set + pHash dups + spider-mites, asserts protocol.
run_mod("model.prepare_plantdoc", "--exclude_set", "model/data/real_world_set", "--artifacts_dir", ARTIFACTS, "--skip_if_exists")
upload("prep", "model/outputs/data_manifest.json", "model/data/plantdoc_train/prep_report.md",
       "model/data/plantdoc_train/prep_report.json", "model/data/plantdoc_train/excluded.csv")


In [ ]:
from IPython.display import Markdown, display
from pathlib import Path
display(Markdown(Path("model/data/plantdoc_train/prep_report.md").read_text()))

### Diagnostics on PlantDoc validation (v1.0, before fine-tuning)

Validation only. Writes `diagnostics_v1_0.json` used later by the publish gate. Crop masking is off.

In [ ]:
run_mod("model.diagnostics", "--model", os.environ["HF_MODEL_ID"], "--revision", "v1.0",
        "--manifest", "model/data/plantdoc_train/manifest.csv",
        "--extra_dir", "model/data/plantdoc_train", "--out", "model/outputs")
upload("diagnostics", "model/outputs/diagnostics_v1_0.json", "model/outputs/diagnostics_v1_0.md")

### Mixed epoch recipe (PlantVillage vs PlantDoc weights)

Before fine-tuning, this cell reads the prepared PlantDoc train labels and shows how `sampling.py`
will draw each mixed epoch: uniform PlantVillage + PlantDoc with class weight `1/sqrt(freq)`.
It does not start training.

In [ ]:
import csv
from pathlib import Path
from IPython.display import display
from notebooks.run import knob
from model.sampling import class_weight_summary

manifest = Path("model/data/plantdoc_train/manifest.csv")
if not manifest.exists():
    print("PlantDoc manifest missing; re-run the 7b prep cell.")
else:
    labels = [r["label"] for r in csv.DictReader(manifest.open()) if r["split"] != "validation"]
    frac = float(knob("PD_FRACTION"))
    n_draw = int(knob("EPOCH_SAMPLES"))
    summary = class_weight_summary(labels, frac, n_draw)
    print(
        f"each epoch: {summary['n_pv_draws']} uniform PlantVillage draws + "
        f"{summary['n_pd_draws']} weighted PlantDoc draws "
        f"(fraction {summary['plantdoc_fraction']}, {summary['epoch_samples']} total)"
    )
    print("PlantDoc classes use weight 1/sqrt(freq); rare classes get more draws. PlantVillage draws are uniform.")
    try:
        import matplotlib.pyplot as plt
        names = [r["class"] for r in summary["classes"]]
        draws = [r["expected_draws"] for r in summary["classes"]]
        fig, ax = plt.subplots(figsize=(10, max(4.0, 0.28 * len(names))))
        ax.barh(names, draws, color="#6b4c9a")
        ax.set_xlabel("Expected PlantDoc draws per epoch")
        ax.set_title("PlantDoc 1/sqrt(freq) sampling")
        ax.invert_yaxis()
        fig.tight_layout()
        out = Path("model/outputs/plantdoc_class_weights.png")
        out.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out, dpi=120)
        display(fig)
        plt.close(fig)
        print("Wrote", out)
    except Exception as e:
        print("Chart skipped:", e)
    print(f"{'class':40s} {'n':>5} {'share':>8} {'exp_draws':>10}")
    for row in summary["classes"]:
        print(f"{row['class']:40s} {row['count']:5d} {row['inv_sqrt_share']:8.4f} {row['expected_draws']:10.1f}")


### Train from `v1.0` (learning rates from the knobs cell, checkpoints on Drive)

Mixed sampler: `PD_FRACTION` PlantDoc (class weight `1/sqrt(freq)`), the rest PlantVillage,
`EPOCH_SAMPLES` draws/epoch, `STAGE2_EPOCHS` epochs, label smoothing 0.1, cosine schedule, PlantDoc RRC (0.5, 1.0).
Best checkpoint by **PlantDoc validation macro-F1**. Resume with `--resume_from_checkpoint auto` after a disconnect.
The next cell trains every learning rate in `STAGE2_LRS` (default 1e-4 and 5e-5) into separate Drive folders.

**Skipped unless** `RUN_STAGE2 = True` in the **section 2b knobs cell**. Set it True to train v1.1.


In [ ]:
import os
from notebooks.run import flag_on, knob, stage2_lrs
from model.common import gpu_train_flags

if not flag_on("RUN_STAGE2"):
    print("Skipping 7b mixed train: Set RUN_STAGE2 = True in the knobs cell to train v1.1.")
else:
    frac = knob("PD_FRACTION")
    n_draw = knob("EPOCH_SAMPLES")
    epochs = knob("STAGE2_EPOCHS")
    batch = knob("STAGE2_BATCH")
    for lr_s, out in zip(stage2_lrs(), os.environ["STAGE2_LR_DIRS"].split(",")):
        print(f"=== Stage 2 train lr={lr_s} -> {out} ===")
        print(f"init_from {os.environ['HF_MODEL_ID']}@v1.0  (PlantVillage weights, then mix in PlantDoc)")
        print(f"mix: {frac} PlantDoc 1/sqrt(freq) + {1 - float(frac):.2f} PlantVillage, "
              f"{n_draw} draws/epoch, {epochs} epochs")
        run_mod(
            "model.train", "--init_from", os.environ["HF_MODEL_ID"], "--model_revision", "v1.0",
            "--data_dir", os.environ["DATA_DIR"], "--extra_data", "plantdoc",
            "--plantdoc_fraction", frac, "--epoch_samples", n_draw, "--artifacts_dir", "model/outputs",
            "--output_dir", out, "--resume_from_checkpoint", "auto",
            "--do_train", "--do_eval", "--num_train_epochs", epochs,
            "--per_device_train_batch_size", batch, "--per_device_eval_batch_size", "128",
            "--learning_rate", lr_s, "--weight_decay", "0.01", "--label_smoothing_factor", "0.1",
            "--lr_scheduler_type", "cosine", *gpu_train_flags(),
            "--eval_strategy", "epoch", "--save_strategy", "epoch", "--save_total_limit", "2",
            "--load_best_model_at_end", "--metric_for_best_model", "plantdoc_macro_f1",
            "--greater_is_better", "True", "--logging_steps", "50", "--report_to", "none", "--seed", "42",
        )
        upload(f"train_lr{lr_s}", f"{out}/run_info.json", f"{out}/eval_results.json")


### Learning-rate folders

Each learning rate in `STAGE2_LRS` has its own Drive folder (`lr1e-4`, `lr5e-5`, …) so a resume does not overwrite the other run. The next cell lists those folders; the cell after that picks the winner on **PlantDoc validation** only.

In [ ]:
import os
from pathlib import Path
from notebooks.run import flag_on, stage2_lrs

print("Stage 2 learning-rate folders (from knobs STAGE2_LRS):")
for lr_s, out in zip(stage2_lrs(), os.environ.get("STAGE2_LR_DIRS", "").split(",")):
    weights = Path(out) / "model.safetensors"
    print(f"  lr {lr_s}: {out}  weights={'yes' if weights.exists() else 'not yet'}")
if not flag_on("RUN_STAGE2"):
    print("Skipping 7b train: Set RUN_STAGE2 = True in the knobs cell to train v1.1.")


In [ ]:
from notebooks.run import flag_on, stage2_lrs

if not flag_on("RUN_STAGE2"):
    print("Skipping 7b compare/evaluate: Set RUN_STAGE2 = True in the knobs cell to train v1.1.")
else:
    import json, os
    from pathlib import Path

    def plantdoc_f1(run):
        p = Path(run) / "eval_results.json"
        if not p.exists():
            p = Path(run) / "all_results.json"
        ev = json.loads(p.read_text())
        return float(ev.get("eval_plantdoc_macro_f1") or ev.get("eval_macro_f1") or 0), ev

    scored = []
    for lr_s, out in zip(stage2_lrs(), os.environ["STAGE2_LR_DIRS"].split(",")):
        f1, ev = plantdoc_f1(out)
        print(f"{lr_s} PlantDoc val macro-F1", f1)
        scored.append((f1, out, lr_s))
    BEST = max(scored, key=lambda row: row[0])[1]
    os.environ["STAGE2_DIR"] = BEST
    print("keeping", BEST)

    run_mod("model.evaluate", "--model", BEST, "--data_dir", os.environ["DATA_DIR"],
            "--reports_dir", f"{BEST}/reports", "--card_revision", "v1.1")
    m = json.load(open(os.path.join(BEST, "reports", "metrics.json")))
    print({k: m[k] for k in ("accuracy", "macro_f1", "top3_accuracy")}, "(v1.0: 0.9912, 0.9858, 0.9994)")
    upload("evaluate", f"{BEST}/reports/metrics.json", f"{BEST}/run_info.json")


### Calibrate on PlantDoc validation

Fits temperature T by NLL on validation logits. Sweeps the live chain (strategy B) 0.30–0.95;
keeps the highest pass rate among thresholds with ≥70% precision of 'ok' answers. If none qualify,
the threshold is **provisional** and the API config is left unchanged.

In [ ]:
from notebooks.run import flag_on

if not flag_on("RUN_STAGE2"):
    print("Skipping 7b calibrate: Set RUN_STAGE2 = True in the knobs cell to train v1.1.")
else:
    run_mod("model.calibrate", "--model", os.environ["STAGE2_DIR"],
            "--manifest", "model/data/plantdoc_train/manifest.csv",
            "--extra_dir", "model/data/plantdoc_train")
    upload("calibrate", f"{os.environ['STAGE2_DIR']}/calibration.json",
           f"{os.environ['STAGE2_DIR']}/threshold.json",
           f"{os.environ['STAGE2_DIR']}/calibration_report.md")

### Final test (once per revision) then publish gate

PlantDoc `test/` plus the 50-image set, frozen T and threshold. Guard: `hub:$HF_MODEL_ID` on the
`stage2-runs` branch (`final_test_guard/v1.1.json`). A second run is refused unless `--force`.
Do not change the threshold or the data after seeing this number.

Publish `v1.1` only if PlantDoc val macro-F1 beats v1.0 **and** PlantVillage accuracy drops by at most 2 points. The live API pin is `shared/release.json` (`v1.1`). **This notebook does not edit that file.**

In [ ]:
from notebooks.run import flag_on, print_publish_checklist

print_publish_checklist()
if not flag_on("RUN_STAGE2"):
    print("Skipping 7b final_test/publish: Set RUN_STAGE2 = True in the knobs cell to train v1.1.")
else:
    run_mod("model.final_test", "--v11", os.environ["STAGE2_DIR"], "--revision_name", "v1.1",
            "--guard", f"hub:{os.environ['HF_MODEL_ID']}",
            "--v10", os.environ["HF_MODEL_ID"], "--v10_revision", "v1.0",
            "--test_dir", "model/data/plantdoc_test", "--real_world", "model/data/real_world_set",
            "--out", "model/outputs")
    upload("final_test", "model/outputs/final_report_v1_1.md", "model/outputs/final_test_per_image.csv")

    run_mod("model.publish", "--model", os.environ["STAGE2_DIR"], "--hub_model_id", os.environ["HF_MODEL_ID"],
            "--tag", "v1.1", "--diagnostics", "model/outputs/diagnostics_v1_0.json", "--dry_run")
    run_mod("model.publish", "--model", os.environ["STAGE2_DIR"], "--hub_model_id", os.environ["HF_MODEL_ID"],
            "--tag", "v1.1", "--diagnostics", "model/outputs/diagnostics_v1_0.json",
            "--tag_message", "Stage 2: PlantVillage + Cropped-PlantDoc",
            "--commit_message", "Stage 2: fine-tune on PlantDoc")


### Hub publish checklist

Three uploads: **weights**, **reports/card**, **release tag**. `shared/release.json` is not written by this notebook.

In [ ]:
from notebooks.run import print_publish_checklist

print_publish_checklist()

## 8. Publish and tag the release for Phase 5

`model/publish.py` uses the `HF_TOKEN` Colab secret: it checks the token and owner, uploads the final model folder
(weights, `config.json`, `splits.json`, `reports/`, model card; no checkpoints) and tags the release.
Run the dry run first if you want to see the file list.

**Skipped unless** `RUN_STAGE1 = True` in the section 2b knobs cell. The live API pin is `shared/release.json` and this notebook does not change it.
`shared/release.json`. This notebook **does not** edit that file. Hub upload is three steps: **weights**
(section 5 `--push_to_hub`), **reports/card** (section 6 `evaluate.py --push`), **tag** (`publish.py --tag v1.0` here).
Stage 2 tagged `v1.1` in section 7b. Do not retag `v1.0` from a stale Drive folder.

In [ ]:
import os
from notebooks.run import ensure_repo, flag_on, print_publish_checklist, require_env, run

ensure_repo()
print_publish_checklist()
if not flag_on("RUN_STAGE1"):
    print("Skipping section 8 publish dry-run: check shared/release.json for the live tag. Set RUN_STAGE1 = True in the knobs cell to retag v1.0 from a fresh train.")
else:
    require_env("OUTPUT_DIR", "HF_MODEL_ID")
    run("model/publish.py", "--model", os.environ["OUTPUT_DIR"], "--hub_model_id", os.environ["HF_MODEL_ID"], "--dry_run")


In [ ]:
import os
from notebooks.run import ensure_repo, flag_on, require_env, run

ensure_repo()
if not flag_on("RUN_STAGE1"):
    print("Skipping section 8 tag v1.0: check shared/release.json for the live tag. Set RUN_STAGE1 = True in the knobs cell to retag v1.0 from a fresh train.")
else:
    require_env("OUTPUT_DIR", "HF_MODEL_ID")
    run("model/publish.py", "--model", os.environ["OUTPUT_DIR"], "--hub_model_id", os.environ["HF_MODEL_ID"], "--tag", "v1.0")


## 9. Inference demo from the Hub (what Phase 5 will do)

In [ ]:
import json, os, random
from PIL import Image as PILImage
from IPython.display import display

import sys
sys.path.insert(0, "model")
from predict import diagnose

from pathlib import Path
pin = json.loads(Path("shared/release.json").read_text())
threshold = pin["confidence_threshold"]
# Live pin: id and revision come from the same file. Never mix HF_MODEL_ID (ResNet) with pin revision (MobileNet).
live_id, live_rev = pin["hf_model_id"], pin["hf_model_revision"]

data_dir = os.environ["DATA_DIR"]
cls = random.choice(sorted(os.listdir(data_dir)))
img_path = os.path.join(data_dir, cls, random.choice(os.listdir(os.path.join(data_dir, cls))))
display(PILImage.open(img_path).resize((224, 224)))
print("True class:", cls)
print("live pin", f"{live_id}@{live_rev}", "threshold", threshold)
print(json.dumps(diagnose(img_path, model_id=live_id, revision=live_rev, threshold=threshold), indent=2))

## 10. Handover to Phase 5

The live API / Space pin is **`shared/release.json`** (copy to `hf_space/release.json`).
`.env` overrides that file; do not scatter Hub tags in `api/app/config.py`.

```env
# Leave HF_MODEL_* unset: the API uses shared/release.json (live pin, MobileNet v1.1).
# Do NOT set HF_MODEL_REVISION=v1.1 with the ResNet repo: that revision is not published yet.
CONFIDENCE_THRESHOLD=0.5
CALIBRATION_TEMPERATURE=0.8845188130635788
CROP_STRATEGY=B
```

- `model/predict.py` is the reference implementation (quality gate -> model -> top-k -> not sure).
- Class names in `config.json` (`id2label`) are PlantVillage folder names and match the keys of
  `api/data/knowledge/diseases.json`.
- Full instructions, challenges and the ResNet fallback are in `model/README.md`.